# Notebook 15 — Mini-Project: Hardy-Weinberg + Genetic Drift Simulator

**Module:** 05 — Biology Fundamentals  
**Notebook:** 15 of 18  
**Prerequisites:** NB06, NB07  
**Portfolio artifact:** `portfolio/module05_hardy_weinberg_drift_simulator.png`

> **Do not look at the implementations in NB06 and NB07 while building this.
> Write from memory, then check. This is Pass-3 reconstruction in practice.**

**Pass standard:** All 4 panels present, labeled, with biological interpretation.
Chi-square test result stated explicitly. Saved to portfolio.

---
## Project Brief

Build a self-contained population genetics simulator from scratch. The final output is
a 4-panel publication-ready figure demonstrating:

- **Panel A:** Allele frequency trajectories showing drift alone (multiple trials)
- **Panel B:** Allele frequency trajectories with directional selection (s = 0.1)
- **Panel C:** Time to fixation vs. population size (N = 10, 50, 200, 1000)
- **Panel D:** HWE expected vs. observed genotype frequencies with chi-square test

Each panel must have:
- Clear axis labels with units
- Title stating the biological interpretation (not just 'Panel A')
- At least one quantitative annotation (e.g. median fixation time)

---
## Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

In [ ]:
# ============================================================
# CORE IMPLEMENTATION — write this from memory first
# ============================================================

def wright_fisher(N: int, p0: float, n_gen: int, n_trials: int,
                  s: float = 0.0, seed: int = 42) -> np.ndarray:
    """
    Wright-Fisher genetic drift simulation.

    Returns: shape (n_trials, n_gen+1) — allele frequency trajectories
    """
    rng = np.random.default_rng(seed)
    trajectories = np.zeros((n_trials, n_gen + 1))
    trajectories[:, 0] = p0

    for t in range(n_gen):
        p = trajectories[:, t]
        # Selection: modify effective frequency before sampling
        if s != 0.0:
            q = 1.0 - p
            p_eff = p * (1 + s) / (p * (1 + s) + q)
        else:
            p_eff = p
        # Genetic drift: binomial sampling
        trajectories[:, t + 1] = rng.binomial(2 * N, p_eff) / (2 * N)

    return trajectories

def time_to_fixation(trajectories: np.ndarray) -> np.ndarray:
    """
    For each trial, return the generation at which p first reaches 0 or 1.
    Returns NaN if still segregating at end.
    """
    n_trials, n_gen_plus1 = trajectories.shape
    fix_times = []
    for trial in trajectories:
        fixed = np.where((trial >= 1.0) | (trial <= 0.0))[0]
        fix_times.append(fixed[0] if len(fixed) > 0 else np.nan)
    return np.array(fix_times)

def hwe_expected(p_hat: float, N: int) -> dict:
    """Expected genotype counts under Hardy-Weinberg."""
    q = 1.0 - p_hat
    return {'AA': p_hat**2 * N, 'Aa': 2*p_hat*q * N, 'aa': q**2 * N}

def hwe_chi2_test(n_AA: int, n_Aa: int, n_aa: int) -> tuple:
    """
    Chi-square test for HWE. Returns (chi2_stat, p_value).
    """
    N = n_AA + n_Aa + n_aa
    p_hat = (2*n_AA + n_Aa) / (2*N)
    exp = hwe_expected(p_hat, N)
    obs = np.array([n_AA, n_Aa, n_aa], dtype=float)
    expv = np.array([exp['AA'], exp['Aa'], exp['aa']])
    chi2 = np.sum((obs - expv)**2 / expv)
    p_val = stats.chi2.sf(chi2, df=1)
    return chi2, p_val, p_hat, exp

print("Core functions defined. Running simulations...")

# ---- Panel A: Drift only
traj_drift = wright_fisher(N=100, p0=0.5, n_gen=300, n_trials=50, s=0.0)

# ---- Panel B: Drift + selection
traj_sel   = wright_fisher(N=100, p0=0.5, n_gen=300, n_trials=50, s=0.10)

# ---- Panel C: Time to fixation vs. N
N_values = [10, 50, 200, 1000]
fix_time_data = {}
for N in N_values:
    n_gen_c = min(4000, 40 * N)   # simulate long enough
    t = wright_fisher(N=N, p0=0.5, n_gen=n_gen_c, n_trials=200, s=0.0)
    fix_time_data[N] = time_to_fixation(t)

# ---- Panel D: HWE test
# Simulate a population reaching HWE from p=0.3 after 100 generations
rng_hwe = np.random.default_rng(1234)
p_hwe = 0.3
N_hwe = 5000
# Sample genotypes from HWE
n_AA_obs = rng_hwe.binomial(N_hwe, p_hwe**2)
n_aa_obs = rng_hwe.binomial(N_hwe - n_AA_obs, (1-p_hwe)**2 / (1-p_hwe**2))
n_Aa_obs = N_hwe - n_AA_obs - n_aa_obs

chi2_stat, chi2_pval, p_hat_hwe, exp_hwe = hwe_chi2_test(n_AA_obs, n_Aa_obs, n_aa_obs)

print(f"Panel A (drift): {(traj_drift[:,-1]==1).sum()} fixed, {(traj_drift[:,-1]==0).sum()} lost")
print(f"Panel B (sel s=0.10): {(traj_sel[:,-1]==1).sum()} fixed")
print(f"Panel D HWE test: χ²={chi2_stat:.3f}, p={chi2_pval:.4f}")

---
## Portfolio Figure

In [ ]:
# 4-panel portfolio figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Population Genetics Simulator: Drift, Selection, and Hardy-Weinberg',
             fontsize=13, fontweight='bold', y=1.01)

gen_axis = np.arange(301)

# ---- Panel A: Drift only
ax = axes[0, 0]
fixed_color, lost_color, seg_color = 'tomato', 'steelblue', 'lightgray'
for trial in traj_drift:
    final = trial[-1]
    color = fixed_color if final >= 1.0 else (lost_color if final <= 0.0 else seg_color)
    ax.plot(gen_axis, trial, color=color, alpha=0.4, lw=0.7)

n_fixed = (traj_drift[:,-1] >= 1.0).sum()
n_lost  = (traj_drift[:,-1] <= 0.0).sum()
ax.set_xlabel('Generation'); ax.set_ylabel('Frequency of allele A')
ax.set_title(f'A. Genetic drift (N=100, p₀=0.5, 50 trials)\n'
             f'Fixed: {n_fixed}  |  Lost: {n_lost}  |  Seg: {50-n_fixed-n_lost}',
             fontsize=10)
ax.axhline(0.5, color='gray', ls='--', lw=0.8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=fixed_color, label='Fixed (p=1)'),
                   Patch(color=lost_color, label='Lost (p=0)'),
                   Patch(color=seg_color, label='Segregating')], fontsize=8)

# ---- Panel B: Selection
ax = axes[0, 1]
for trial_d, trial_s in zip(traj_drift[:20], traj_sel[:20]):
    ax.plot(gen_axis, trial_d, color='steelblue', alpha=0.3, lw=0.8)
    ax.plot(gen_axis, trial_s, color='tomato', alpha=0.3, lw=0.8)
ax.plot([], [], color='steelblue', label='Drift only (s=0)', lw=2)
ax.plot([], [], color='tomato', label='Positive selection (s=0.1)', lw=2)

# Plot means
ax.plot(gen_axis, traj_drift.mean(axis=0), color='navy', lw=2.5, label='Mean drift')
ax.plot(gen_axis, traj_sel.mean(axis=0), color='darkred', lw=2.5, label='Mean + selection')
ax.set_xlabel('Generation'); ax.set_ylabel('Frequency of allele A')
n_sel_fixed = (traj_sel[:,-1] >= 1.0).sum()
ax.set_title(f'B. Selection (s=0.1) drives fixation\n'
             f'{n_sel_fixed}/50 fixed under selection vs. {n_fixed}/50 by drift alone', fontsize=10)
ax.legend(fontsize=7)

# ---- Panel C: Fixation time vs. N
ax = axes[1, 0]
fix_medians = []
for N in N_values:
    ft = fix_time_data[N]
    valid = ft[~np.isnan(ft)]
    fix_medians.append(np.median(valid) if len(valid) > 0 else np.nan)

ax.plot(N_values, fix_medians, 'o-', color='seagreen', lw=2, ms=8)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Population size N (log scale)')
ax.set_ylabel('Median generations to fixation (log scale)')
ax.set_title('C. Larger populations fix neutral alleles more slowly\n'
             'Expected: fixation time ∝ 4N generations', fontsize=10)
for N, med in zip(N_values, fix_medians):
    if not np.isnan(med):
        ax.annotate(f'{med:.0f} gen', (N, med), textcoords='offset points',
                    xytext=(8, 4), fontsize=8)
# Theory line: t_fix ≈ 4N
N_theory = np.array([10, 1000])
ax.plot(N_theory, 4 * N_theory, 'k--', lw=1, alpha=0.5, label='4N (theory)')
ax.legend(fontsize=8)

# ---- Panel D: HWE test
ax = axes[1, 1]
genotypes = ['AA', 'Aa', 'aa']
observed = [n_AA_obs, n_Aa_obs, n_aa_obs]
expected_v = [exp_hwe['AA'], exp_hwe['Aa'], exp_hwe['aa']]

x = np.arange(3)
bars_obs = ax.bar(x - 0.2, observed, 0.35, label='Observed', color='steelblue', alpha=0.8)
bars_exp = ax.bar(x + 0.2, expected_v, 0.35, label='HWE expected', color='lightgray',
                  edgecolor='steelblue', linewidth=1.5)
ax.set_xticks(x); ax.set_xticklabels(genotypes, fontsize=11)
ax.set_ylabel('Count')
hwe_result = 'IN HWE' if chi2_pval >= 0.05 else 'DEVIATES from HWE'
ax.set_title(f'D. Hardy-Weinberg test (N={N_hwe}, p̂={p_hat_hwe:.3f})\n'
             f'χ²={chi2_stat:.2f}, p={chi2_pval:.4f} → {hwe_result}', fontsize=10)
ax.legend()

# Annotate differences
for xi, (obs, exp) in enumerate(zip(observed, expected_v)):
    ax.annotate(f'Δ={obs-exp:+.0f}', (xi, max(obs, exp) + 10),
                ha='center', fontsize=8, color='gray')

plt.tight_layout()

# Save to portfolio
portfolio_dir = '../../../../../portfolio'
os.makedirs(portfolio_dir, exist_ok=True)
out_path = os.path.join(portfolio_dir, 'module05_hardy_weinberg_drift_simulator.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f"Saved to {out_path}")
plt.show()

---
## Self-Assessment Checklist

Before marking this notebook complete, verify:

- [ ] All 4 panels are present with biological interpretation in the title
- [ ] Panel A shows allele frequency trajectories with fixation/loss counts
- [ ] Panel B shows the effect of selection compared to drift
- [ ] Panel C shows fixation time scaling with N (compare to 4N theoretical prediction)
- [ ] Panel D states the chi-square value and p-value explicitly
- [ ] Figure saved to `portfolio/module05_hardy_weinberg_drift_simulator.png`
- [ ] Can explain every line of code without looking at NB06/NB07

---
## Reflection

**Date completed:** ____________________

> *[What was the hardest part to implement from memory? What does the discrepancy
> between observed fixation time and 4N theory tell you about the N=10 case?]*

---
*Next: `16_mini_project_biology_for_cs.ipynb`*